# Silverbox — estimação no multisseno e validação na seta

Execute no VS Code com o mesmo kernel usado no CED. **Este notebook foi preparado sem executar o download ou a identificação aqui.**

A divisão segue a ideia da figura que você enviou: estimação no trecho de amplitude aproximadamente constante à direita e validação na seta inteira à esquerda. **Os cortes abaixo são os limites de segmentos do carregador oficial v1.0.1, não índices confirmados no texto do artigo da figura.** Não reaplicamos a divisão 75/25 do carregador: todo o segmento multisseno selecionado é usado para estimação.

| Uso | Fatia Python (fim exclusivo) | Índices MATLAB equivalentes |
|---|---|---|
| Estimação: multisseno à direita | `[40650:127400]` | `40651:127400` |
| Validação: seta inteira à esquerda | `[100:40575]` | `101:40575` |

A primeira célula de configuração permite alterar esses limites. Não há normalização adicional nem decimação. As primeiras 50 amostras **da própria seta** inicializam a simulação, e as restantes entram nas métricas. Não usamos o fim do treino como histórico, porque treino e validação são segmentos separados.

A configuração inicial é FROLS + mínimos quadrados, grau 3, 14 atrasos de entrada/saída, até 80 termos e `err_tol=0.9999`, como no caso Arrow full do [exemplo online do livro](https://sysidentpy.org/user-guide/tutorials/silver-box-system/). **A nova divisão de treino modifica o experimento:** os coeficientes e erros não precisam coincidir com os resultados do livro.

Fonte dos dados e cortes: [carregador nonlinear_benchmarks](https://github.com/MaartenSchoukens/nonlinear_benchmarks), função `Silverbox` da versão 1.0.1. Entrada e saída são tensões medidas do circuito, em volts.


In [ ]:
%pip install sysidentpy==0.9.0 nonlinear-benchmarks==1.0.1 numpy scipy pandas matplotlib


In [ ]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import platform
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import savemat
from IPython.display import display
import nonlinear_benchmarks
from sysidentpy.model_structure_selection import FROLS
from sysidentpy.basis_function import Polynomial
from sysidentpy.parameter_estimation import LeastSquares
from sysidentpy.metrics import root_mean_squared_error

OUT = Path("Silverbox_arrow_validation_outputs").resolve()
OUT.mkdir(parents=True, exist_ok=True)
VERSIONS = {p: version(p) for p in ["sysidentpy", "nonlinear-benchmarks", "numpy", "scipy", "pandas", "matplotlib"]}
VERSIONS["python"] = platform.python_version()
print(json.dumps(VERSIONS, indent=2))
print("Pasta de saída:", OUT)

# Limites Python: início inclusivo, fim exclusivo.
TRAIN_START = 40650
TRAIN_STOP = 127400
VALID_START = 100
VALID_STOP = 40575
N_INITIAL = 50

# Um modelo de referência, usando o trecho multisseno inteiro para estimação.
RUN_CASES = ["arrow_full"]
CONFIGS = {
    "arrow_full": dict(degree=3, lag=14, err_tol=0.9999, n_terms=80),
}
# Altere degree, lag, err_tol ou n_terms acima apenas se quiser outra configuração.


## 1. Dados e verificação visual da divisão

Carregamos o registro completo com `train_test_split=False` e aplicamos as fatias declaradas acima. O gráfico marca os trechos usados, os intervalos não utilizados e o início das amostras efetivamente avaliadas. A seta permanece contínua: não removemos amostras em função de sua amplitude.

O arquivo `Silverbox_arrow_full_data.mat` preserva os dados de estimação e validação, os índices no registro original, o período de amostragem e a janela inicial. O carregador pode reutilizar o download em cache.


In [ ]:
raw = nonlinear_benchmarks.Silverbox(train_test_split=False, atleast_2d=True)
u_raw = np.asarray(raw.u, dtype=float).reshape(-1, 1)
y_raw = np.asarray(raw.y, dtype=float).reshape(-1, 1)
assert u_raw.shape == y_raw.shape
assert np.isfinite(u_raw).all() and np.isfinite(y_raw).all()
assert 0 <= TRAIN_START < TRAIN_STOP <= len(y_raw)
assert 0 <= VALID_START < VALID_STOP <= len(y_raw)
assert TRAIN_STOP <= VALID_START or VALID_STOP <= TRAIN_START, "Treino e validação se sobrepõem."
assert 0 < N_INITIAL < VALID_STOP-VALID_START
assert CONFIGS["arrow_full"]["lag"] <= N_INITIAL

u_train = u_raw[TRAIN_START:TRAIN_STOP].copy()
y_train = y_raw[TRAIN_START:TRAIN_STOP].copy()
d = dict(u_train=u_train, y_train=y_train,
         u_test=u_raw[VALID_START:VALID_STOP].copy(),
         y_test=y_raw[VALID_START:VALID_STOP].copy(),
         n_initial=N_INITIAL, Ts=float(raw.sampling_time),
         train_indices_matlab=np.arange(TRAIN_START+1, TRAIN_STOP+1).reshape(-1,1),
         test_indices_matlab=np.arange(VALID_START+1, VALID_STOP+1).reshape(-1,1),
         train_start_python=TRAIN_START, train_stop_python_exclusive=TRAIN_STOP,
         valid_start_python=VALID_START, valid_stop_python_exclusive=VALID_STOP)
DATA = {"arrow_full": d}
DATA_HASHES = {"arrow_full": {
    k: hashlib.sha256(np.asarray(d[k],dtype="<f8").tobytes()).hexdigest()
    for k in ["u_train","y_train","u_test","y_test"]}}
savemat(OUT / "Silverbox_arrow_full_data.mat",d,do_compression=True)
print(f"Estimação: {len(y_train)} amostras")
print(f"Validação: {len(d['y_test'])} amostras; {N_INITIAL} para inicialização")
print(f"Métricas: {len(d['y_test'])-N_INITIAL} amostras")

k_raw = np.arange(len(y_raw))
fig, axes = plt.subplots(2,1,figsize=(14,7),constrained_layout=True)
for ax, signal, label in [(axes[0],u_raw,"Input voltage (V)"), (axes[1],y_raw,"Output voltage (V)")]:
    ax.plot(k_raw,signal.ravel(),color="#087dbb",linewidth=0.6)
    ax.axvspan(VALID_START,VALID_STOP,color="orange",alpha=0.15,label="Validation — arrow")
    ax.axvspan(TRAIN_START,TRAIN_STOP,color="green",alpha=0.12,label="Estimation — multisine")
    ax.axvline(VALID_START+N_INITIAL,color="darkorange",linestyle=":",label="Scoring starts")
    ax.set(xlabel="Índice no registro original (Python)",ylabel=label)
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
fig.savefig(OUT / "Silverbox_estimation_validation_split.png",dpi=180)
plt.show()


## 2. Identificar e exportar a referência

A identificação usa **somente o multisseno selecionado**. As saídas medidas da seta são usadas apenas para inicialização e cálculo de métricas; não são realimentadas durante a simulação livre.

Grau 3 e 14 atrasos resultam em 4495 monômios candidatos. Com 80750 amostras, uma única matriz float64 ocupa cerca de 2.9 GB, e o ajuste pode criar várias matrizes. Feche outros processos pesados antes de executar. Reduzir grau ou atrasos muda a configuração do modelo; não fazemos isso automaticamente.

Métricas: `RMSE` em volts; `RMSE_mV`; `NRMSE_std = RMSE/std(y avaliado, ddof=0)`; `NRMSE_range`; e `R2`. O `NRMSE_std` é equivalente à definição usada no seu código Green.

A equação exportada é avaliada por uma rotina independente e comparada à previsão do SysIdentPy. O `.mat` contém os coeficientes completos, sem arredondamento de tabelas, e a previsão de referência para futura verificação no MATLAB.


In [ ]:
def regressor_names(codes):
    names = []
    for row in np.asarray(codes):
        factors = []
        for c in row:
            c = int(c)
            if c == 0:
                continue
            signal, lag = divmod(c, 1000)
            assert signal in (1, 2), f"Código inesperado para Silverbox SISO: {c}"
            factors.append(f"{'y' if signal == 1 else 'u'}(k-{lag})")
        names.append(" * ".join(factors) if factors else "1")
    return names

def simulate_exported(codes, theta, u, y_initial):
    # Avaliação independente da equação exportada; mesma inicialização do predict.
    u = np.asarray(u).ravel()
    y_initial = np.asarray(y_initial).ravel()
    y = np.zeros(len(u), dtype=float)
    y[:len(y_initial)] = y_initial
    for k in range(len(y_initial), len(u)):
        total = 0.0
        for row, coefficient in zip(codes, np.asarray(theta).ravel()):
            term = 1.0
            for c in row:
                c = int(c)
                if c:
                    signal, lag = divmod(c, 1000)
                    term *= y[k-lag] if signal == 1 else u[k-lag]
            total += coefficient * term
        y[k] = total
    return y.reshape(-1, 1)


In [ ]:
models = {}
summary_rows = []
for name in RUN_CASES:
    cfg = CONFIGS[name]
    d = DATA[name]
    key = f"Silverbox_{name}_degree{cfg['degree']}_lag{cfg['lag']}"
    print(f"Identificando {key}...", flush=True)
    model = FROLS(xlag=cfg["lag"], ylag=cfg["lag"],
                  basis_function=Polynomial(degree=cfg["degree"]),
                  estimator=LeastSquares(), err_tol=cfg["err_tol"],
                  n_terms=cfg["n_terms"], order_selection=False)
    t0 = time.perf_counter()
    model.fit(X=d["u_train"], y=d["y_train"])
    fit_seconds = time.perf_counter() - t0
    lag = int(model.max_lag)
    n = d["n_initial"]
    if lag > n:
        raise ValueError("Atraso maior que a janela de inicialização do benchmark.")
    start = n - lag
    yhat_full = model.predict(X=d["u_test"][start:], y=d["y_test"][start:n])
    yhat = np.asarray(yhat_full)[lag:].reshape(-1, 1)
    target = d["y_test"][n:]
    assert yhat.shape == target.shape
    if not np.isfinite(yhat).all():
        raise RuntimeError(f"Simulação não finita: {key}")
    codes = np.asarray(model.final_model, dtype=np.int64)
    theta = np.asarray(model.theta).reshape(-1, 1)
    replay = simulate_exported(codes, theta, d["u_test"][start:], d["y_test"][start:n])[lag:]
    np.testing.assert_allclose(replay, yhat, rtol=1e-6, atol=1e-8,
                               err_msg="Equação exportada diverge do predict.")
    replay_max_abs_diff = float(np.max(np.abs(replay-yhat)))
    rmse = float(root_mean_squared_error(target, yhat))
    std = float(np.std(target))
    span = float(np.ptp(target))
    sst = float(np.sum((target-np.mean(target))**2))
    metrics = dict(RMSE=rmse, RMSE_mV=1000*rmse,
                   NRMSE_std=rmse/std if std else float("nan"),
                   NRMSE_range=rmse/span if span else float("nan"),
                   R2=1-float(np.sum((target-yhat)**2))/sst if sst else float("nan"))
    labels = regressor_names(codes)
    table = pd.DataFrame(dict(regressor=labels, coefficient=theta.ravel()))
    table.to_csv(OUT / f"{key}_terms.csv", index=False, float_format="%.17g")
    equation = "y(k) = " + " + ".join(f"({v:.17g}) * {r}" for r,v in zip(labels,theta.ravel()))
    (OUT / f"{key}_equation.txt").write_text(equation + "\n", encoding="utf-8")
    savemat(OUT / f"{key}_model.mat", {
        "theta": theta, "model_code": codes,
        "regressor_labels": np.asarray(labels,dtype=object).reshape(-1,1),
        "test_name": name, "max_lag": lag, "degree": cfg["degree"], "n_terms": len(theta),
        "n_initial": n, "Ts": d["Ts"],
        "raw_evaluation_start_matlab": VALID_START+n+1,
        "raw_train_start_matlab": TRAIN_START+1, "raw_train_end_matlab": TRAIN_STOP,
        "raw_test_start_matlab": VALID_START+1, "raw_test_end_matlab": VALID_STOP,
        "evaluation_start_matlab": n+1, "simulation_start_matlab": start+1,
        "y_initial": d["y_test"][start:n],
        "y_test_evaluated": target, "yhat_test": yhat,
        "replay_max_abs_difference": replay_max_abs_diff, **metrics,
    }, do_compression=True)
    models[name] = dict(model=model, yhat=yhat, target=target, key=key, metrics=metrics)
    summary_rows.append(dict(test=name, model=key, n_terms=len(theta), max_lag=lag,
                             fit_seconds=fit_seconds, replay_max_abs_difference=replay_max_abs_diff, **metrics))
    print(f"{len(theta)} termos; RMSE={rmse:.9g} V; NRMSE_std={metrics['NRMSE_std']:.9g}")
    print(f"Diferença máxima na verificação da equação: {replay_max_abs_diff:.3g}")
    with pd.option_context("display.max_rows", None, "display.precision", 16):
        display(table)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "Silverbox_summary.csv", index=False, float_format="%.17g")
display(summary)
manifest = dict(source="nonlinear_benchmarks.Silverbox, official loader v1.0.1",
                book_url="https://sysidentpy.org/user-guide/tutorials/silver-box-system/",
                versions=VERSIONS, configurations={k:CONFIGS[k] for k in RUN_CASES},
                data_hashes=DATA_HASHES, normalization="none added",
                split=dict(train_start=TRAIN_START,train_stop_exclusive=TRAIN_STOP,
                           validation_start=VALID_START,validation_stop_exclusive=VALID_STOP,
                           n_initial=N_INITIAL),
                split_note="Full multisine segment for estimation; full arrow for validation; article cut not confirmed",
                evaluation="Free run after loader-defined initialization window",
                NRMSE_std_definition="RMSE / std(y_test[n_initial:], ddof=0)")
(OUT / "Silverbox_run_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")


In [ ]:
for name, result in models.items():
    d = DATA[name]
    n = d["n_initial"]
    k = np.arange(VALID_START+n,VALID_START+n+len(result["target"]))
    fig, axes = plt.subplots(2,1,figsize=(14,7),constrained_layout=True)
    for ax, count, title in [(axes[0],len(k),"Validação — seta completa"), (axes[1],min(300,len(k)),"Primeiras 300 amostras avaliadas")]:
        ax.plot(k[:count], result["target"][:count].ravel(), color="black", linewidth=3.5, label="Data")
        ax.plot(k[:count], result["yhat"][:count].ravel(), color="#822327", linewidth=2,
                marker="o" if count <= 300 else None, markevery=5, markersize=8,
                markerfacecolor="none", label="SysIdentPy")
        ax.set(title=f"{name} — {title}", xlabel="Índice no registro original (Python)", ylabel="Output voltage (V)")
        ax.grid(alpha=0.25)
        ax.legend()
    fig.suptitle(f"{result['key']} — RMSE={result['metrics']['RMSE_mV']:.4f} mV")
    fig.savefig(OUT / f"{result['key']}_comparison.png",dpi=180)
    plt.show()
print("Concluído. Arquivos em:",OUT)


## 3. Arquivos para usar no MATLAB

Os principais arquivos estarão em `Silverbox_arrow_validation_outputs`:

- `Silverbox_arrow_full_data.mat`: dados de treino e validação, com seus índices.
- `Silverbox_arrow_full_degree3_lag14_model.mat`: modelo fixo, previsão e métricas.
- `Silverbox_arrow_full_degree3_lag14_terms.csv`: termos e coeficientes.
- `Silverbox_arrow_full_degree3_lag14_equation.txt`: equação legível.
- `Silverbox_summary.csv`: resumo dos resultados.
- `Silverbox_run_manifest.json`: versões, configuração, cortes e hashes dos dados.

```matlab
D = load('Silverbox_arrow_validation_outputs/Silverbox_arrow_full_data.mat');
M = load('Silverbox_arrow_validation_outputs/Silverbox_arrow_full_degree3_lag14_model.mat');
u_id = D.u_train;
y_id = D.y_train;
u_val = D.u_test;
y_val = D.y_test;
reference.theta = M.theta;
reference.model_code = M.model_code;
```

Os nomes dos arquivos do modelo acompanham degree/lag se você mudar a configuração.

**Codificação:** cada linha de `model_code` corresponde à mesma linha de `theta`. Os fatores são `0` = 1, `1000+lag` = saída atrasada e `2000+lag` = entrada atrasada. A equação não precisa ser escrita manualmente.

**Alinhamento:** `M.yhat_test` corresponde a `D.y_test(M.evaluation_start_matlab:end)`. Esse índice é relativo ao segmento de validação; `M.raw_evaluation_start_matlab` é o índice no registro original. Use os últimos `M.max_lag` valores dentro das primeiras `D.n_initial` amostras da seta para inicializar a referência. Para comparar Green, mantenha a mesma janela de avaliação e o mesmo protocolo de inicialização.

Este notebook não foi executado aqui, conforme solicitado. Antes de reportar resultados, confira o gráfico da divisão e a verificação da equação após executá-lo. Escolher pesos/modelos com base nesta validação a torna parte da seleção; não a descreva então como um teste final independente.
